In [ ]:
from pathlib import Path
import pandas as pd
from dtale import show
import dtale.global_state as global_state

global_state.set_app_settings(dict(max_column_width=300))

data_dir = Path().absolute() / ".." / "data"
df = pd.read_json(data_dir / "conversations/cm0i27jdj0000aqpa73ghpcxf.json")

In [ ]:
import pandas as pd
from datetime import datetime

def process_conversation_data(original_df):
    """
    Process conversation data into a structured DataFrame with conversation_id,
    date, time, and content columns.
    
    Args:
        original_df (pd.DataFrame): DataFrame containing the mapping column
    
    Returns:
        pd.DataFrame: Processed conversation data
    """
    # Initialize lists to store the processed data
    processed_data = []
    
    # Iterate through each row in the original DataFrame
    for conversation_id, mapping in original_df["mapping"].items():
        conversation_data = list(mapping.values())
        
        for message in conversation_data:
            # Check if the message has required fields and is from a user
            if (message.get('message') and 
                message['message'].get('author') and 
                message['message']['author'].get('role') == 'user' and 
                message['message'].get('create_time') and 
                message['message'].get('content')):
                
                # Extract timestamp and convert to datetime
                timestamp = message['message']['create_time']
                dt = datetime.fromtimestamp(timestamp)
                
                # Extract content (assuming it's in the 'parts' list)
                content = message['message']['content'].get('parts', [''])[0]
                
                processed_data.append({
                    'conversation_id': conversation_id,
                    'date': dt.date(),
                    'time': dt.time(),
                    'content': content
                })
    
    # Create DataFrame from processed data
    df_conversations = pd.DataFrame(processed_data)
    
    # Sort by date and time
    df_conversations = df_conversations.sort_values(['date', 'time'])
    
    return df_conversations

# Example usage:
messasges_df = process_conversation_data(df)  # where df is your original DataFrame

d = show(messasges_df)
d.open_browser()

In [ ]:
import networkx as nx
from typing import List, Dict, Any
import pandas as pd
import matplotlib.pyplot as plt

def create_conversation_dag(conversation_data: List[Dict[Any, Any]]) -> nx.DiGraph:
    """
    Creates a Directed Acyclic Graph from conversation data structure.
    
    Args:
        conversation_data: List of dictionaries containing message data with 
                         'id', 'parent', and 'children' fields
    
    Returns:
        nx.DiGraph: A NetworkX directed graph representing the conversation structure
    """
    # Create a new directed graph
    G = nx.DiGraph()
    
    # Process each message in the conversation data
    for message in conversation_data:
        msg_id = message['id']
        parent_id = message['parent']
        children = message.get('children', [])
        
        # Add the current node
        role = (
            (message.get("message", {}) or {})
            .get("author", {})
            .get("role", "unknown")
        )
        content = (
            (message.get("message", {}) or {})
            .get("content", {})
            .get("parts", ["", ])[0]
        )
        G.add_node(msg_id, role=role, content=content)
        
        # Add edge from parent to current node (if parent exists)
        if parent_id:
            G.add_edge(parent_id, msg_id)
        
        # Add edges to children
        for child_id in children:
            G.add_edge(msg_id, child_id)
    
    return G

def visualize_conversation_dag(G: nx.DiGraph, conversation_idx: int = 0, figsize=(12, 8)):
    """
    Visualize the conversation DAG with different colors for different roles.
    
    Args:
        G: NetworkX directed graph to visualize
        conversation_idx: Index of the conversation being visualized
        figsize: Tuple specifying figure dimensions
    """
    plt.figure(figsize=figsize)
    
    # Create a layout for the graph
    pos = nx.spring_layout(G, k=2, iterations=50)
    
    # Define colors for different roles
    role_colors = {
        'system': 'gray',
        'user': 'lightblue',
        'assistant': 'lightgreen',
        'unknown': 'lightgray'
    }
    
    # Draw nodes with different colors based on role
    for role in role_colors:
        node_list = [node for node, attr in G.nodes(data=True) 
                    if attr.get('role') == role]
        nx.draw_networkx_nodes(G, pos, 
                             nodelist=node_list,
                             node_color=role_colors[role],
                             node_size=1000)
    
    # Draw edges and labels
    nx.draw_networkx_edges(G, pos, edge_color='gray', arrows=True)
    nx.draw_networkx_labels(G, pos, labels={node: node[:8] + '...' 
                                          for node in G.nodes()})
    
    plt.title(f'Conversation {conversation_idx + 1} Structure DAG')
    plt.axis('off')
    plt.tight_layout()
    
def analyze_dag(G: nx.DiGraph) -> Dict:
    """
    Analyze the DAG and return useful metrics.
    
    Args:
        G: NetworkX directed graph to analyze
    
    Returns:
        Dict containing various metrics about the graph
    """
    # Find root nodes (nodes with no incoming edges)
    root_nodes = [n for n in G.nodes() if G.in_degree(n) == 0]
    
    # Calculate max depth from each root node
    max_depths = []
    for root in root_nodes:
        try:
            depths = nx.shortest_path_length(G, source=root)
            max_depths.append(max(depths.values()))
        except nx.NetworkXError:
            continue
    
    return {
        'total_nodes': G.number_of_nodes(),
        'total_edges': G.number_of_edges(),
        'max_depth': max(max_depths) if max_depths else 0,
        'leaf_nodes': len([n for n in G.nodes() if G.out_degree(n) == 0]),
        'root_nodes': len(root_nodes),
        'is_dag': nx.is_directed_acyclic_graph(G)
    }

def process_conversation_data(df: pd.DataFrame) -> List[nx.DiGraph]:
    """
    Process conversation data from a DataFrame and create visualizations for all conversations.
    
    Args:
        df: DataFrame containing a 'mapping' column with conversation structures
    
    Returns:
        List[nx.DiGraph]: List of all conversation DAGs
    """
    # Convert DataFrame column to list of conversations
    all_conversations = list(map(lambda x: list(x.values()), df["mapping"].to_list()))
    all_dags = []
    
    # Process each conversation
    for idx, conversation_data in enumerate(all_conversations):
        # Create and analyze the DAG
        G = create_conversation_dag(conversation_data)
        metrics = analyze_dag(G)
        all_dags.append(G)
        
        print(f"\nConversation {idx + 1} Metrics:")
        for metric, value in metrics.items():
            print(f"{metric}: {value}")
        
        # Visualize the DAG
        visualize_conversation_dag(G, idx)
        plt.show()
    
    return all_dags


In [ ]:
import networkx as nx
from typing import List, Dict, Any
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import os
import re
from datetime import datetime

def create_output_directory() -> str:
    """
    Create a uniquely named output directory for this run.
    
    Returns:
        str: Path to the created directory
    """
    # Create base directory for all conversation plots if it doesn't exist
    base_dir = "conversations_viz"
    os.makedirs(base_dir, exist_ok=True)
    
    # Create a unique directory name using timestamp
    #timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    #output_dir = os.path.join(base_dir, f"analysis_{timestamp}")
    output_dir = base_dir
    os.makedirs(output_dir)
    
    print(f"Created output directory: {output_dir}")
    return output_dir

def save_dag_plot(G: nx.DiGraph, title: str, output_dir: str, figsize=(800, 600)):
    """
    Create and save an interactive visualization for a single DAG.
    
    Args:
        G: NetworkX DiGraph
        title: Title of the conversation (used for filename)
        output_dir: Directory to save the plot
        figsize: Tuple of (width, height) for the figure
    """
    # Create a safe filename from title
    filename = re.sub(r'[^\w\s-]', '', title).strip().lower()
    filename = re.sub(r'[-\s]+', '-', filename)
    
    # Create figure
    fig = go.Figure()
    
    # Define colors for different roles
    role_colors = {
        'system': '#808080',
        'user': '#ADD8E6',
        'assistant': '#90EE90',
        'unknown': '#D3D3D3'
    }
    
    try:
        # Create layout for this graph
        pos = nx.spring_layout(G, k=1/np.sqrt(len(G.nodes())), iterations=50)
        
        # Create edges
        edge_x = []
        edge_y = []
        for edge in G.edges():
            x0, y0 = pos[edge[0]]
            x1, y1 = pos[edge[1]]
            edge_x.extend([x0, x1, None])
            edge_y.extend([y0, y1, None])
            
        # Add edges to plot
        fig.add_trace(
            go.Scatter(
                x=edge_x, 
                y=edge_y,
                line=dict(width=0.5, color='#888'),
                hoverinfo='none',
                mode='lines',
                name='connections'
            )
        )
        
        # Process nodes by role
        for role in role_colors:
            node_list = [node for node, attr in G.nodes(data=True) 
                        if attr.get('role') == role]
            if not node_list:
                continue
                
            node_x = [pos[node][0] for node in node_list]
            node_y = [pos[node][1] for node in node_list]
            
            # Create hover text with full message content
            hover_text = []
            for node in node_list:
                content = G.nodes[node].get('content', '')
                formatted_content = ''
                
                if content:
                    # Convert content to string and split into lines
                    content_str = str(content)
                    lines = []
                    for i in range(0, len(content_str), 80):
                        lines.append(content_str[i:i+80])
                    formatted_content = '<br>'.join(lines)
                
                hover_text.append(
                    f"Role: {role}<br>"
                    f"ID: {str(node)}<br>"
                    f"Content:<br>{formatted_content}"
                )
            
            # Add nodes to plot
            fig.add_trace(
                go.Scatter(
                    x=node_x, 
                    y=node_y,
                    mode='markers+text',
                    marker=dict(
                        size=20,
                        color=role_colors[role],
                        line=dict(width=1, color='#666')
                    ),
                    text=[str(node)[:8] + '...' for node in node_list],
                    textposition="bottom center",
                    hovertext=hover_text,
                    hoverinfo='text',
                    hoverlabel=dict(
                        bgcolor="white",
                        font_size=12,
                        font_family="monospace",
                        align="left"
                    ),
                    name=role
                )
            )
        
        # Update layout
        fig.update_layout(
            title=dict(
                text=title,
                x=0.5,
                xanchor='center',
                yanchor='top'
            ),
            showlegend=True,
            width=figsize[0],
            height=figsize[1],
            paper_bgcolor='white',
            plot_bgcolor='white',
            legend=dict(
                yanchor="top",
                y=0.99,
                xanchor="left",
                x=0.01
            ),
            margin=dict(l=50, r=50, t=50, b=50)
        )
        
        # Update axes
        fig.update_xaxes(showgrid=False, zeroline=False, showticklabels=False)
        fig.update_yaxes(showgrid=False, zeroline=False, showticklabels=False)
        
        # Save the plot
        plot_path = os.path.join(output_dir, f'{filename}.html')
        fig.write_html(plot_path)
        print(f"Saved plot to {plot_path}")
        
        # Save metrics to a text file
        metrics_path = os.path.join(output_dir, f'{filename}_metrics.txt')
        with open(metrics_path, 'w') as f:
            metrics = analyze_dag(G)
            f.write(f"Metrics for {title}\n")
            f.write("=" * (len(title) + 12) + "\n\n")
            for metric, value in metrics.items():
                f.write(f"{metric}: {value}\n")
        
    except Exception as e:
        print(f"Error processing DAG for {title}: {str(e)}")
        raise  # Re-raise the exception to see the full traceback

def process_conversation_data_interactive(df: pd.DataFrame) -> List[nx.DiGraph]:
    """
    Process conversation data and save individual visualizations for each conversation.
    
    Args:
        df: DataFrame containing 'mapping' column
    
    Returns:
        List[nx.DiGraph]: List of all conversation DAGs
    """
    # Create output directory for this run
    output_dir = create_output_directory()
    
    # Convert DataFrame column to list of conversations
    all_conversations = list(map(lambda x: list(x.values()), df["mapping"].to_list()))
    all_dags = []
    
    print(f"Processing {len(all_conversations)} conversations")
    
    # Save summary info
    with open(os.path.join(output_dir, "analysis_summary.txt"), 'w') as f:
        f.write(f"Conversation Analysis Summary\n")
        f.write("=========================\n\n")
        f.write(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Total conversations: {len(all_conversations)}\n\n")
    
    # Process each conversation
    for idx, conversation_data in enumerate(all_conversations):
        # Create and analyze the DAG
        G = create_conversation_dag(conversation_data)
        all_dags.append(G)
        
        # Generate title (use DataFrame title if available, otherwise use index)
        title = f"Conversation {idx + 1}"
        if 'title' in df.columns:
            title = str(df['title'].iloc[idx])
        
        print(f"\nProcessing {title}")
        
        # Save individual plot
        save_dag_plot(G, title, output_dir)
    
    print(f"\nAnalysis complete. Results saved in: {output_dir}")
    return all_dags

In [ ]:
process_conversation_data_interactive(df)